# Pré-remplissage — la référence verbatim, proposée par CrisperWhisper

Il transcrit les 36 prises ; la référence s'écrit ensuite **à l'oreille contre sa
proposition**, prise par prise. Ce n'est pas lui la mesure : c'est un brouillon.

**Licence** : CrisperWhisper est sous licence non commerciale. Elle interdit de le
**livrer**, pas de s'en servir pour mesurer. Rien de ce modèle n'entre dans l'app.

Il passe aussi sur les 24 prises écrites, dont la référence est connue au mot près.
C'est gratuit et ça vaut un contrôle : s'il rate des hésitations **jouées**, son
brouillon des réponses spontanées ne vaut pas grand-chose non plus.

Réglages Kaggle : accélérateur **GPU T4**, Internet **on**, persistance *Files only*.
Dataset à attacher : `gilleslandrin/saylune-hesitation-takes`.

Ce qui rentre à la maison : `prefill.json`, quelques kilo-octets.

In [ ]:
!pip install -q "crisperwhisper[ct2]"

In [ ]:
import json, pathlib

from crisperwhisper import CrisperWhisperModel

model = CrisperWhisperModel()
print("loaded")

In [ ]:
root = next(p for p in pathlib.Path("/kaggle/input").iterdir() if p.is_dir())
references = json.loads((root / "references.json").read_text(encoding="utf-8"))
asked = json.loads((root / "answers" / "asked.json").read_text(encoding="utf-8"))
takes = sorted(root.rglob("*.wav"))
print(len(takes), "takes under", root)

In [ ]:
def said(result):
    """The transcribed text, whatever shape the result comes back in."""
    if isinstance(result, dict):
        return result.get("text", result)
    return getattr(result, "text", result)


rows = []
for wav in takes:
    which = wav.parent.name
    slug = wav.stem
    verbatim = said(model.transcribe(str(wav), language="en"))
    intended = said(model.transcribe(str(wav), language="en", mode="intended"))
    row = {"set": which, "slug": slug, "verbatim": verbatim, "intended": intended}
    if which == "stumbles":
        row["reference"] = references[slug]["text"]
        row["kind"] = references[slug]["kind"]
    else:
        row["asked"] = asked.get(slug)
    rows.append(row)
    print(f"\n[{which}/{slug}]")
    if "reference" in row:
        print(f"  attendu : {row['reference']}")
    print(f"  verbatim: {verbatim}")
    print(f"  intended: {intended}")

In [ ]:
out = pathlib.Path("/kaggle/working/prefill.json")
out.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(out, out.stat().st_size, "bytes")